# Technique: data poisoning with image data

Based on the following paper:

**Goodfellow, Ian J., Jonathon Shlens, and Christian Szegedy. "Explaining and harnessing adversarial examples." arXiv preprint arXiv:1412.6572 (2014).**

In this project we'll examine some popular attacks which apply data-poisoning methods to maximize the loss of a target neural network.

This technique (FGSM) takes in test data and modifies the individual pixel values according to parameter $\epsilon \in [0,1]$ and a normalization scheme. We'll compare the accuracy between the in-class CNN and the victim CNN.

In [32]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
import numpy as np
import torch.nn.functional as F

'''
CNN.ipynb
----------

Implements a CNN (using the implementation from class with some tweaks.) Performs
rudimentary data analysis using the MNIST dataset, and then attempts to perturb
the dataset so our CNN will misclassify the image. The perturbations should be as minor
and undetectable as possible.

We want to search for a perturbation which does not heinously alter our image.
Rotations and translations, pixel displacements, etc.

Fast Gradient Sign Attack which maximizes loss with respect to input data

'''
def LpNorm():
  '''
  Lp-norm based attack
  '''
  return
def fgsm(loss_gradient, img_data, perturbation_parameter):
  '''
  Takes in perturbation parameter epsilon and returns an
  altered image coordinate. This is a linear-style attack which
  can be applied to nonlinear data.
  '''
  # get sign of loss gradient
  sign_loss_gradient = loss_gradient.sign()

  # pixel-wise adjustment
  new_img_data = img_data + perturbation_parameter * sign_loss_gradient

  # clipping
  new_img_data = torch.clamp(new_img_data, 0, 1)

  return new_img_data

def denorm(batch, mean=[0.1307], std=[0.3081]):
  if isinstance(mean, list):
    mean = torch.tensor(mean).to(device)
  if isinstance(std, list):
    std = torch.tensor(std).to(device)

  return batch * std.view(1, -1, 1, 1) + mean.view(1,-1, 1, 1)

def perturbator(model, device, test_loader, epsilon):
  '''
  This is a bad-faith test function which injects modified test data into the dataset.
  '''
  ex = []
  # Calculate Accuracy
  correct = 0
  # Iterate through test dataset
  for images, labels in test_loader:

    # Load images
    images,labels = images.to(device), labels.to(device)
    images.requires_grad = True

    # Forward pass only to get logits/output
    outputs = model(images)

    # Get predictions from the max log-probability
    init_predicted = outputs.max(1, keepdim=True)[1]

    if predicted.item() != labels.item():
      continue

    loss = F.nll_loss(outputs, labels)

    model.zero_grad()

    loss.backward()

    data_grad = images.grad.data

    data_denorm = denorm(images)

    # mount the attack
    pdata = fgsm(data_grad, data_denorm, epsilon)

    # normalize perturbed data
    pdata_normalized = transforms.Normalize((0.1307), (0.3081))(pdata)

    outputs = model(pdata_normalized)

    final_predicted = outputs.max(1, keepdim=True)[1]
    if final_predicted.item() == labels.item():
      correct += 1
      if epsilon == 0 and len(ex) < 5:
        adv_ex = pdata.squeeze().detach().cpu().numpy()
        ex.append((init_predicted.item(), final_predicted.item(), adv_ex))
    else:
      if len(ex) < 5:
        adv_ex = pdata.squeeze().detach().cpu().numpy()
        ex.append((init_predicted.item(), final_predicted.item(), ex))

  final_accuracy = correct/float(len(test_loader))

  # Print loss
  print('Epsilon_val: {}. Loss: {}. Accuracy: {}'.format(epsilon, loss.item(), accuracy))
  return final_accuracy, adv_ex

train_dataset = dsets.MNIST(root='./data',
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

test_dataset = dsets.MNIST(root='./data',
                           train=False,
                           transform=transforms.ToTensor())

# make dataset iterable
batch_size = 1
n_iters = 3000
num_epochs = 2
num_epochs = int(num_epochs)
print(num_epochs)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=100,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=True)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# this will be the NN that we'll be attacking
class victim_CNN(nn.Module):
  def __init__(self):
    super(victim_CNN, self).__init__()

    # Convolution 1
    self.cnn1 = nn.Conv2d(in_channels = 1, out_channels = 16, kernel_size=5, stride=1, padding=2)
    self.relu1 = nn.ReLU()

    # Max pool 1
    self.maxpool1 = nn.MaxPool2d(kernel_size = 2)

    # Convolution 2
    self.cnn2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 5, stride = 1, padding = 2)
    self.relu2 = nn.ReLU()

    # Max pool 2
    self.maxpool2 = nn.MaxPool2d(kernel_size = 2)

    self.fc1 = nn.Linear(32 * 7 * 7, 10)

  def forward(self, x):
    # input: x, size (num_img, 28, 28)

    # Convolution 1
    # O = (28 - 5 + 2*2)/ 1 + 1 = 28
    # output: size (num_img, 16, 28, 28)
    out = self.cnn1(x)
    out = self.relu1(out)

    # Max pool 1
    # O = 28 / 2 = 14
    # output: size (num_img, 16, 14, 14)
    out = self.maxpool1(out)

    # Convolution 2
    # O = (14 - 5 + 2*2)/ 1 + 1 = 14
    # output: size (num_img, 32, 14, 14)
    out = self.cnn2(out)
    out = self.relu2(out)

    # Max pool 2
    # O = 14 / 2 = 7
    # output: size (num_img, 32, 7, 7)
    out = self.maxpool2(out)

    # Resize
    # Original size: (num_img, 32, 7, 7)
    # out.size(0): num_img
    # New out size: (num_img, 32*7*7)
    out = out.view(out.size(0), -1)

    # Linear function (readout)
    # output: size (num_img, 10)
    out = self.fc1(out)

    # log softmax
    out = F.log_softmax(out, dim=1)

    return out

model = victim_CNN()
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# now let us train the dataset. We'll test intermittently (this is perfectly customizable!)
iter = 0
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Load images
        images = images.requires_grad_()

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()

        # Forward pass to get output/logits
        outputs = model(images)

        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)

        # Getting gradients w.r.t. parameters
        loss.backward()

        # Updating parameters
        optimizer.step()

        iter += 1

        if iter % 500 == 0:
            # Calculate Accuracy
            correct = 0
            total = 0
            # Iterate through test dataset
            for images, labels in test_loader:
                # Load images
                images = images.requires_grad_()

                # Forward pass only to get logits/output
                outputs = model(images)

                # Get predictions from the maximum value
                _, predicted = torch.max(outputs.data, 1)

                # Total number of labels
                total += labels.size(0)

                # Total correct predictions
                correct += (predicted == labels).sum()

            accuracy = 100 * correct / total

            # Print Loss
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))

            if iter == n_iters:
              break




2
Iteration: 500. Loss: 0.32116591930389404. Accuracy: 88.66000366210938
Iteration: 1000. Loss: 0.27902933955192566. Accuracy: 93.0199966430664


## Inject the bad data into the trained dataset


In [33]:
total_accuracy = []
total_ex = []
epsilons = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]
for epsilon in epsilons:
  accuracy, ex = perturbator(model, device, test_loader, epsilon)
  total_accuracy.append(accuracy)
  total_ex.append(ex)

Epsilon_val: 0. Loss: 0.025686707347631454. Accuracy: 93.0199966430664
Epsilon_val: 0.1. Loss: 0.36847344040870667. Accuracy: 0.0912
Epsilon_val: 0.2. Loss: 0.31152674555778503. Accuracy: 0.0
Epsilon_val: 0.3. Loss: 0.041122548282146454. Accuracy: 0.0
Epsilon_val: 0.4. Loss: 4.583194255828857. Accuracy: 0.0
Epsilon_val: 0.5. Loss: 4.386902809143066. Accuracy: 0.0
Epsilon_val: 0.6. Loss: 0.0012528197839856148. Accuracy: 0.0
Epsilon_val: 1.0. Loss: 0.00319886626675725. Accuracy: 0.0
